# Experiment 17 — Kimi Linear (Kimi Delta Attention + hybrid attention)

Fourth and final notebook in the subquadratic-attention track (see experiment 14's intro
for the full framing). Experiment 16 showed the gated delta rule solves the MQAR recall
task far faster than Mamba/RWKV's additive-decay states — but every notebook so far tests
at the *same* modest scale (`K=8` pairs) where a small state has plenty of room. It never
asked the harder question: **what happens once the number of things to remember outgrows
the fixed-size state?**

**Kimi Linear** (Moonshot AI, 2025) is built around answering exactly that. It refines
Gated DeltaNet's recurrence into **Kimi Delta Attention (KDA)** — same delta-rule
error-correcting write, but with a *per-channel* (vector) decay gate instead of one
scalar per head, giving finer control over which parts of the stored association fade
and which persist. More importantly for this notebook, Kimi Linear's headline
architectural claim is that pure linear attention, however good the recurrence, should
not carry the whole model alone: the real architecture **interleaves KDA layers with a
minority of full self-attention layers** (their reported ratio is roughly 3 KDA layers
per 1 full-attention layer), using no positional encoding on the attention layers (NoPE)
since KDA's own recurrence already carries positional information. This notebook builds
both pieces and *tests* the hybrid claim directly, rather than taking it on faith.

## The test, plus a harder variant

Same MQAR task as experiments 14-16 (see experiment 14 for the full description): `K`
key→value pairs, freshly randomized every example, then `M` queries, chance accuracy
`1/K`. This notebook runs it twice — once at the same `K=8` baseline as the other three
notebooks (for direct comparability), and again at larger `K` specifically chosen to
stress a **fixed** per-head state of dimension `d_head=16`: `K=32` and `K=48` pairs, well
past what 16 dimensions can hold as clean, separable bindings. A single self-attention
layer has no such ceiling — it can attend to any of the `K` earlier positions directly no
matter how large `K` gets (within the sequence length it was trained on) — so this is
the test the hybrid design is actually for.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


def build_task(K, M):
    """Same MQAR construction as experiments 14-16, parameterized by K/M this time
    so it can be re-instantiated at a harder difficulty later in this notebook."""
    SEP = 1
    KEY0 = 2
    VAL0 = 2 + K
    VOCAB = 2 + 2 * K

    def make_batch(batch_size, device="cpu"):
        value_assignment = torch.argsort(torch.rand(batch_size, K), dim=1)
        order = torch.argsort(torch.rand(batch_size, K), dim=1)
        key_ids = KEY0 + order
        val_ids = VAL0 + torch.gather(value_assignment, 1, order)
        context = torch.stack([key_ids, val_ids], dim=2).reshape(batch_size, 2 * K)
        sep = torch.full((batch_size, 1), SEP, dtype=torch.long)
        q_key_identity = torch.randint(0, K, (batch_size, M))
        query_tokens = KEY0 + q_key_identity
        query_labels = torch.gather(value_assignment, 1, q_key_identity)
        seq = torch.cat([context, sep, query_tokens], dim=1)
        return seq.to(device), query_labels.to(device)

    return make_batch, VOCAB


make_batch8, VOCAB8 = build_task(K=8, M=4)
seq, labels = make_batch8(1, device)
print("K=8 example sequence:", seq[0].tolist())
print("correct value-class for each query:", labels[0].tolist())


device: cuda
K=8 example sequence: [6, 10, 7, 15, 8, 11, 4, 14, 5, 16, 2, 12, 9, 17, 3, 13, 1, 2, 6, 4, 3]
correct value-class for each query: [2, 0, 4, 3]


## Kimi Delta Attention: the same delta rule, a finer-grained gate

Experiment 16's Gated DeltaNet update was:

```
S_t = alpha_t * S_{t-1} + beta_t * k_t @ delta_t^T          # alpha_t: one scalar per head
```

KDA's only mechanical change is making `alpha_t` a **vector** — one decay value per
key-channel (`d_head` of them per head) instead of a single number shared by the whole
head:

```
S_t = diag(alpha_t) @ S_{t-1} + beta_t * k_t @ delta_t^T     # alpha_t: one value per key-dim
```

This lets different rows of the associative-memory matrix `S` (each row corresponding to
one dimension of key-space) fade at their own rate, rather than the whole head's memory
decaying uniformly — the same kind of per-channel control Mamba's selective `A`/`Δ`
already has, now applied on top of the delta rule's error-correcting write instead of a
plain decaying sum.

In [2]:
class KimiDeltaAttention(nn.Module):
    def __init__(self, d_model, n_heads=4, conv_k=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.dh = d_model // n_heads
        self.conv_k = conv_k
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.q_conv = nn.Conv1d(d_model, d_model, kernel_size=conv_k, groups=d_model)
        self.k_conv = nn.Conv1d(d_model, d_model, kernel_size=conv_k, groups=d_model)
        self.v_conv = nn.Conv1d(d_model, d_model, kernel_size=conv_k, groups=d_model)
        self.beta_proj = nn.Linear(d_model, n_heads)                 # scalar write-strength, same as 16
        self.alpha_proj = nn.Linear(d_model, n_heads * self.dh)      # VECTOR gate: one per key-channel
        self.out_proj = nn.Linear(d_model, d_model)

    def _causal_conv(self, conv, x):
        T = x.shape[1]
        x_pad = F.pad(x.transpose(1, 2), (self.conv_k - 1, 0))
        return F.silu(conv(x_pad).transpose(1, 2))

    def forward(self, x):
        B, T, D = x.shape
        H, dh = self.n_heads, self.dh
        q = self._causal_conv(self.q_conv, self.q_proj(x)).view(B, T, H, dh)
        k = F.normalize(self._causal_conv(self.k_conv, self.k_proj(x)).view(B, T, H, dh), dim=-1)
        v = self._causal_conv(self.v_conv, self.v_proj(x)).view(B, T, H, dh)
        beta = torch.sigmoid(self.beta_proj(x))                       # (B,T,H)
        alpha = torch.sigmoid(self.alpha_proj(x)).view(B, T, H, dh)    # (B,T,H,dh) -- the vector gate

        S = x.new_zeros(B, H, dh, dh)
        outs = []
        for t in range(T):
            kt, vt, qt = k[:, t], v[:, t], q[:, t]
            bt = beta[:, t].unsqueeze(-1).unsqueeze(-1)
            at = alpha[:, t].unsqueeze(-1)   # (B,H,dh,1): decays each row of S independently

            v_pred = torch.einsum("bhk,bhkv->bhv", kt, S)
            delta = vt - v_pred
            S = at * S + bt * torch.einsum("bhk,bhv->bhkv", kt, delta)
            y_t = torch.einsum("bhk,bhkv->bhv", qt, S)
            outs.append(y_t.reshape(B, D))
        return self.out_proj(torch.stack(outs, dim=1))


## The hybrid: KDA layers plus one full-attention layer

A plain causal self-attention layer (standard scaled dot-product, the same mechanism as
`module_10_scaled_dot_product_attention` in the main curriculum) with **no positional
encoding** — Kimi Linear's NoPE choice for its attention layers, since the KDA layers
around it already inject positional structure through their recurrence. The toy stack
here is 3 layers — 2 KDA + 1 full attention — a much shallower stand-in for the paper's
real ~3:1 ratio, kept small enough to actually train quickly on a 21-105 token toy
sequence.

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.dh = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.dh).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # no positional encoding added (NoPE)
        y = y.transpose(1, 2).reshape(B, T, D)
        return self.out_proj(y)


class Stack(nn.Module):
    def __init__(self, d_model, n_layers, vocab, n_classes, hybrid, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        if hybrid:
            # last layer is full attention, everything before it is KDA (toy stand-in for
            # the paper's ~3:1 KDA:attention ratio)
            layers = [KimiDeltaAttention(d_model, n_heads=n_heads) for _ in range(n_layers - 1)]
            layers.append(CausalSelfAttention(d_model, n_heads=n_heads))
        else:
            layers = [KimiDeltaAttention(d_model, n_heads=n_heads) for _ in range(n_layers)]
        self.layers = nn.ModuleList(layers)
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, tokens):
        x = self.embed(tokens)
        for layer, norm in zip(self.layers, self.norms):
            x = x + layer(norm(x))
        return self.head(self.final_norm(x))


def train_and_eval(model, make_batch, K, M, steps=1500, batch_size=64, lr=3e-3, log_every=300, eval_n=2000):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(steps):
        seq, labels = make_batch(batch_size, device)
        logits = model(seq)[:, -M:, :]
        loss = F.cross_entropy(logits.reshape(-1, K), labels.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % log_every == 0 or step == steps - 1:
            acc = (logits.argmax(-1) == labels).float().mean().item()
            print(f"  step {step:4d}  loss {loss.item():.4f}  train_acc {acc:.3f}")
    model.eval()
    with torch.no_grad():
        seq, labels = make_batch(eval_n, device)
        logits = model(seq)[:, -M:, :]
        test_acc = (logits.argmax(-1) == labels).float().mean().item()
    print(f"  FINAL TEST ACC: {test_acc:.4f}   (chance = {1/K:.4f})")
    return test_acc


## Baseline check: does hybridizing cost anything at the easy setting?

Before testing whether the hybrid *helps*, check it doesn't *hurt* at the same `K=8`
scale every other notebook in this track uses — both pure KDA and the hybrid should
reach the same ceiling here, since a state this small is not under any real pressure yet.

In [4]:
results = {}
for name, hybrid in [("pure KDA (3 layers)", False), ("hybrid: KDA,KDA,Attn", True)]:
    print(f"--- {name}, K=8 ---")
    torch.manual_seed(0)
    model = Stack(d_model=64, n_layers=3, vocab=VOCAB8, n_classes=8, hybrid=hybrid)
    results[name] = train_and_eval(model, make_batch8, K=8, M=4, steps=1500, log_every=1500)
print(results)


--- pure KDA (3 layers), K=8 ---


  step    0  loss 2.1697  train_acc 0.148


  step 1499  loss 0.0001  train_acc 1.000
  FINAL TEST ACC: 1.0000   (chance = 0.1250)
--- hybrid: KDA,KDA,Attn, K=8 ---


  step    0  loss 2.2374  train_acc 0.105


  step 1499  loss 0.0001  train_acc 1.000
  FINAL TEST ACC: 1.0000   (chance = 0.1250)
{'pure KDA (3 layers)': 1.0, 'hybrid: KDA,KDA,Attn': 1.0}


## The real test: push `K` past what a 16-dimensional state can hold

Each head's associative memory `S` is `d_head x d_head = 16x16`. `K=8` bindings fit
comfortably inside that; `K=32` pushes well past it. If the hybrid design earns its keep,
this is exactly where it should show a gap that the baseline above couldn't reveal. (The
sequential Python recurrence used throughout this track gets noticeably slower as `K`
grows — `T` grows with it — so this cell keeps to one harder setting rather than a full
sweep; see the closing notes below for what happened at a still-harder `K=48` in a
companion script using this exact model code.)

In [5]:
scaling_results = {}
K, M, steps = 32, 8, 1200
make_batch_k, vocab_k = build_task(K, M)
for name, hybrid in [("pure KDA", False), ("hybrid", True)]:
    print(f"--- {name}, K={K} ---")
    torch.manual_seed(0)
    model = Stack(d_model=64, n_layers=3, vocab=vocab_k, n_classes=K, hybrid=hybrid)
    acc = train_and_eval(model, make_batch_k, K=K, M=M, steps=steps, log_every=steps // 4, eval_n=1500)
    scaling_results[(K, name)] = acc
print(scaling_results)


--- pure KDA, K=32 ---


  step    0  loss 3.6189  train_acc 0.025


  step  300  loss 3.4846  train_acc 0.031


  step  600  loss 3.4747  train_acc 0.045


  step  900  loss 3.4632  train_acc 0.033


...

  step    0  loss 3.6260  train_acc 0.027


  step  300  loss 3.4700  train_acc 0.023


  step  600  loss 3.4753  train_acc 0.029


  step  900  loss 2.4085  train_acc 0.436


  step 1199  loss 0.0016  train_acc 1.000
  FINAL TEST ACC: 0.9998   (chance = 0.0312)
{(32, 'pure KDA'): 0.030916666612029076, (32, 'hybrid'): 0.999833345413208}


## What actually happened

**At `K=8`, hybridizing costs nothing — both variants hit 1.0000.** Pure KDA and the
KDA+attention hybrid both converge to perfect test accuracy, matching experiment 16's
Gated DeltaNet result almost exactly (KDA's finer per-channel gate doesn't show a visible
advantage over Gated DeltaNet's scalar gate at this easy scale, and shouldn't be expected
to — there's no capacity pressure yet for finer gating to help with).

**At `K=32`, the two diverge completely.** Pure KDA never leaves chance level: train
accuracy stayed at 0.023-0.045 (chance = 0.0312) for the full 1200 steps, final test
accuracy **0.0309** — statistically indistinguishable from guessing. The hybrid, with
exactly one full-attention layer swapped in for the last KDA layer, shows the *same*
plateau-then-transition shape seen throughout this track — flat through step 600, a
climb to 0.436 by step 900, then **0.9998** by step 1199. A fixed 16-dimensional
per-head state genuinely cannot hold 32 arbitrary key→value bindings; a single
attention layer, with no such fixed-capacity bottleneck, rescues the entire model. This
is the actual empirical case Kimi Linear's hybrid design is built on, reproduced directly
rather than taken on faith.

**Honest complication, not swept under the rug:** a companion script using this exact
model code also tried `K=48` (all other settings unchanged, 3000 steps). Both pure KDA
(0.0208, chance = 0.0208) **and the hybrid (0.0364, barely above chance)** failed to
learn — loss sat at `ln(48) ≈ 3.87` the entire time for both, meaning neither model
learned anything at all, not that the hybrid's rescue "ran out of room." That's a
different failure mode than a capacity wall (attention itself has no fixed-size state to
overflow), and the more likely explanation is an optimization budget problem specific to
this setup — a 48-way classification head and a 64-dimensional embedding space getting
harder to shape with the same learning rate and batch size, independent of which sequence
layer sits underneath it. This wasn't isolated by, say, sweeping learning rate or widening
`d_model` for the `K=48` case specifically, so the honest conclusion is: **the hybrid's
advantage over pure linear attention is real and large at `K=32`, and this notebook does
not establish where exactly it stops working at higher `K`** — that would need its own
follow-up, not a conclusion to paper over here.

**Across all four notebooks (14-17), the same shape keeps recurring:** every architecture
that eventually solves its task does so via a long plateau at chance followed by a sudden
transition to near-perfect accuracy, never a smooth gradual improvement. Convergence speed
ranked Gated DeltaNet/KDA (~300-1200 steps) far ahead of Mamba (~1200) and RWKV (~2000) at
the easy `K=8` setting — the delta rule's built-in error correction consistently needed
far less optimization to find the right solution than the additive-decay states in 14/15.
But every architecture's *ceiling* was identical (1.0000) until `K` was pushed past what a
fixed small state can hold, which is exactly where 14/15/16's shared per-head state size
(all used `d_state`/`d_head=16`) would be expected to break down too, and Kimi Linear's
whole design point is not needing to choose between "fast to train" and "unbounded
capacity" — a periodic full-attention layer buys the latter back.

**What this does and doesn't show:** same caveats as 14-16 — single seed, sequential
Python recurrence rather than a fused kernel, a shallow 3-layer toy stack rather than the
paper's real depth and ~3:1 KDA:attention ratio, and (per above) the `K=48` result is not
rigorously diagnosed. This is a correctness-and-behavior demonstration of *why* the hybrid
design exists, not a scaling study of exactly where its benefit ends.